# 授業方略実験ノートブック

最終更新: 2026-07-14 10:30 JST

このノートブックでは、教師AIが授業手法を考える前段階を確認します。生徒AIの発話を作り、伝達AIが観察し、教師AI用のコンテキストを作成して、次の授業方略を選びます。

## 1. GitHubから最新版を取得

Colabでは最初にこのセルを実行します。まだcloneしていない場合はcloneし、すでにclone済みの場合は最新版をpullします。

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/Hiromu-0219/student-ai-test.git"
PROJECT_DIR = Path("/content/student-ai")

if not PROJECT_DIR.exists():
    !git clone {REPO_URL} {PROJECT_DIR}

%cd /content/student-ai
!git status --short --branch
!git pull
!git log -1 --oneline
!grep -n "summarize_classroom" src/observer/trait_classifier.py
!grep -n "classroom_observation" src/teacher/context_builder.py
!grep -n "TeacherBeliefManager" src/teacher/belief_manager.py
!grep -n "RuleBasedInterventionPlanner" src/teacher/intervention_planner.py
!grep -n "RuleBasedTeacherUtteranceBuilder" src/teacher/utterance_builder.py

## 2. 依存関係をインストール

In [ ]:
!pip install -q -r requirements.txt

## 3. 実験設定

軽く確認する場合は `USE_MOCK_MODEL = True` のまま使います。実LLMで生徒発話を生成したい場合は `False` に変更します。

In [ ]:
STUDENT_ID = "S002"
USE_MOCK_MODEL = True
TEACHER_MESSAGE = "x + 3 = 8 を解いてみましょう。+3を反対側へ移すと、符号はどうなりますか。"

print("student_id:", STUDENT_ID)
print("use_mock_model:", USE_MOCK_MODEL)
print("teacher_message:", TEACHER_MESSAGE)

## 4. 生徒AIの発話を生成

ここでは授業中の対話として生徒発話を生成します。実験を繰り返しやすくするため、標準では知識状態を更新しません。

In [ ]:
from src.student_ai import StudentAISimulator

sim = StudentAISimulator(use_mock_model=USE_MOCK_MODEL)
student_record = sim.respond(
    STUDENT_ID,
    TEACHER_MESSAGE,
    update_knowledge=False,
)
student_utterance = student_record["answer"]

print(student_utterance)

## 5. 伝達AIが発話を観察

伝達AIが、生徒発話から見える個人特徴を推定し、教師AIに渡す注意点を作ります。

In [ ]:
import importlib
from pprint import pprint

import src.observer.trait_classifier as trait_classifier

importlib.reload(trait_classifier)
CommunicationAI = trait_classifier.CommunicationAI

communication_ai = CommunicationAI()
print("has summarize_classroom:", hasattr(communication_ai, "summarize_classroom"))
observation = communication_ai.classify_utterance(
    utterance=student_utterance,
    student_id=STUDENT_ID,
).to_dict()

pprint(observation)

## 6. 複数生徒を実際に反応させて要約

S001、S002、S003を同じ教師発話に反応させます。伝達AIには内部状態ではなく、授業中に観察できる発話、正誤、反応時間などだけを渡します。

In [ ]:
import importlib
import time
from pprint import pprint

from src.grader import LinearEquationGrader
import src.observer.observation_filter as observation_filter

importlib.reload(observation_filter)
build_observable_event = observation_filter.build_observable_event
events_to_communication_rows = observation_filter.events_to_communication_rows

CLASS_STUDENT_IDS = ["S001", "S002", "S003"]
EXPECTED_ANSWER = "x = 5"
grader = LinearEquationGrader()

classroom_records = []
observable_events = []

for sid in CLASS_STUDENT_IDS:
    started = time.perf_counter()
    record = sim.respond(sid, TEACHER_MESSAGE, update_knowledge=False)
    response_time_sec = round(time.perf_counter() - started, 2)
    utterance = record["answer"]
    grade = grader.grade(EXPECTED_ANSWER, utterance)
    event = build_observable_event(
        lesson_id="L001",
        teacher_id="T001",
        student_id=sid,
        teacher_prompt=TEACHER_MESSAGE,
        utterance=utterance,
        answer=utterance,
        is_correct=grade["is_correct"],
        response_time_sec=response_time_sec,
    ).to_dict()
    classroom_records.append({"student_id": sid, "utterance": utterance, "grade": grade})
    observable_events.append(event)

classroom_rows = events_to_communication_rows(observable_events)
classroom_observation = communication_ai.summarize_classroom(classroom_rows).to_dict()

pprint({
    "classroom_records": classroom_records,
    "student_count": classroom_observation["student_count"],
    "profile_counts": classroom_observation["profile_counts"],
    "trait_level_counts": classroom_observation["trait_level_counts"],
    "priority_students": classroom_observation["priority_students"],
    "classroom_summary": classroom_observation["classroom_summary"],
    "recommended_teacher_actions": classroom_observation["recommended_teacher_actions"],
})

## 7. 複数生徒の教師側推定を更新して表で確認

各生徒の `teacher_belief` を、観察イベントと伝達AIの個別分類結果から更新します。表では推定理解度、confidence、性格・心理推定の変化を見ます。

In [ ]:
import importlib
import pandas as pd
from pprint import pprint

import src.teacher.belief_manager as belief_manager

importlib.reload(belief_manager)

TeacherBeliefManager = belief_manager.TeacherBeliefManager

belief_manager_instance = TeacherBeliefManager()
teacher_beliefs = belief_manager_instance.update_many(
    teacher_id="T001",
    observations=observable_events,
    communication_results=classroom_observation["individual_results"],
)
teacher_belief = teacher_beliefs[STUDENT_ID]

belief_rows = []
for sid, belief in teacher_beliefs.items():
    knowledge = belief["estimated_knowledge"]["linear_equation"]
    traits = belief["estimated_traits"]
    event = next(item for item in observable_events if item["student_id"] == sid)
    belief_rows.append({
        "student_id": sid,
        "is_correct": event["is_correct"],
        "response_time_sec": event["response_time_sec"],
        "estimated_score": knowledge["score"],
        "score_confidence": knowledge["confidence"],
        "self_efficacy": traits["self_efficacy"]["level"],
        "self_efficacy_conf": traits["self_efficacy"]["confidence"],
        "question_tendency": traits["question_tendency"]["level"],
        "motivation": traits["motivation"]["level"],
        "neuroticism": traits["neuroticism"]["level"],
        "evidence_count": len(belief["evidence_history"]),
    })

belief_table = pd.DataFrame(belief_rows).sort_values("student_id")
display(belief_table)
pprint(teacher_beliefs)

## 8. 複数回授業で推定が蓄積されるか検証

L001〜L003の3回分の授業を同じ3人に実行し、教師側推定のスコア、confidence、履歴数がどう変化するかを確認します。この検証では、既存の `data/teacher_beliefs/T001` を汚さないように検証用ディレクトリへ保存します。

In [ ]:
import importlib
import shutil
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from src.grader import LinearEquationGrader
import src.observer.observation_filter as observation_filter
import src.teacher.belief_manager as belief_manager

importlib.reload(observation_filter)
importlib.reload(belief_manager)

build_observable_event = observation_filter.build_observable_event
events_to_communication_rows = observation_filter.events_to_communication_rows
TeacherBeliefManager = belief_manager.TeacherBeliefManager

validation_dir = Path("data/teacher_beliefs_validation")
if validation_dir.exists():
    shutil.rmtree(validation_dir)

validation_belief_manager = TeacherBeliefManager(validation_dir)
validation_grader = LinearEquationGrader()
validation_lessons = [
    {"lesson_id": "L001", "prompt": "x + 3 = 8 を解いてください。", "expected": "x = 5"},
    {"lesson_id": "L002", "prompt": "2x + 3 = 11 を解いてください。", "expected": "x = 4"},
    {"lesson_id": "L003", "prompt": "3x = 15 を解いてください。", "expected": "x = 5"},
]

belief_progress_rows = []
validation_events_all = []
validation_classroom_summaries = []

for lesson in validation_lessons:
    lesson_events = []
    for sid in CLASS_STUDENT_IDS:
        started = time.perf_counter()
        record = sim.respond(sid, lesson["prompt"], update_knowledge=False)
        response_time_sec = round(time.perf_counter() - started, 2)
        utterance = record["answer"]
        grade = validation_grader.grade(lesson["expected"], utterance)
        event = build_observable_event(
            lesson_id=lesson["lesson_id"],
            teacher_id="T001",
            student_id=sid,
            teacher_prompt=lesson["prompt"],
            utterance=utterance,
            answer=utterance,
            is_correct=grade["is_correct"],
            response_time_sec=response_time_sec,
        ).to_dict()
        lesson_events.append(event)
        validation_events_all.append(event)

    lesson_rows = events_to_communication_rows(lesson_events)
    lesson_summary = communication_ai.summarize_classroom(lesson_rows).to_dict()
    validation_classroom_summaries.append({
        "lesson_id": lesson["lesson_id"],
        "classroom_summary": lesson_summary["classroom_summary"],
        "profile_counts": lesson_summary["profile_counts"],
    })
    updated_beliefs = validation_belief_manager.update_many(
        teacher_id="T001",
        observations=lesson_events,
        communication_results=lesson_summary["individual_results"],
    )

    for sid, belief in updated_beliefs.items():
        knowledge = belief["estimated_knowledge"]["linear_equation"]
        traits = belief["estimated_traits"]
        event = next(item for item in lesson_events if item["student_id"] == sid)
        belief_progress_rows.append({
            "lesson_id": lesson["lesson_id"],
            "student_id": sid,
            "is_correct": event["is_correct"],
            "estimated_score": knowledge["score"],
            "score_confidence": knowledge["confidence"],
            "self_efficacy": traits["self_efficacy"]["level"],
            "question_tendency": traits["question_tendency"]["level"],
            "motivation": traits["motivation"]["level"],
            "neuroticism": traits["neuroticism"]["level"],
            "evidence_count": len(belief["evidence_history"]),
        })

belief_progress_table = pd.DataFrame(belief_progress_rows).sort_values(["student_id", "lesson_id"])
display(belief_progress_table)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for sid, group in belief_progress_table.groupby("student_id"):
    axes[0].plot(group["lesson_id"], group["estimated_score"], marker="o", label=sid)
    axes[1].plot(group["lesson_id"], group["score_confidence"], marker="o", label=sid)
axes[0].set_title("Estimated knowledge score")
axes[0].set_ylim(0, 100)
axes[1].set_title("Score confidence")
axes[1].set_ylim(0, 1)
for ax in axes:
    ax.set_xlabel("lesson")
    ax.grid(True, alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()

display(pd.DataFrame(validation_classroom_summaries))

## 9. クラス全体対応と個別対応を計画

ここではLLMを使わず、クラス要約、教師側belief、直近の観察イベントから、全体向けの授業行動と個別支援を分けて計画します。

In [ ]:
import importlib
from pprint import pprint

import src.teacher.context_builder as teacher_context_builder
import src.teacher.intervention_planner as intervention_planner

importlib.reload(teacher_context_builder)
importlib.reload(intervention_planner)
TeacherContextBuilder = teacher_context_builder.TeacherContextBuilder
RuleBasedInterventionPlanner = intervention_planner.RuleBasedInterventionPlanner
planner_context_builder = TeacherContextBuilder()

latest_lesson_id = validation_lessons[-1]["lesson_id"]
latest_events = [event for event in validation_events_all if event["lesson_id"] == latest_lesson_id]
latest_classroom_observation = communication_ai.summarize_classroom(
    events_to_communication_rows(latest_events)
).to_dict()
latest_teacher_beliefs = {
    sid: validation_belief_manager.load_or_create("T001", sid)
    for sid in CLASS_STUDENT_IDS
}

intervention_plan = RuleBasedInterventionPlanner().plan(
    classroom_observation=latest_classroom_observation,
    teacher_beliefs=latest_teacher_beliefs,
    lesson_goal={
        "target_skill": "can_divide_by_coefficient",
        "goal_text": "係数で両辺を割って x を求める",
    },
    recent_events=latest_events,
    next_problem_bank=planner_context_builder.curriculum["next_problem_bank"],
)

pprint(intervention_plan)
display(pd.DataFrame(intervention_plan["individual_supports"]))

## 10. 授業方略を教師発話に変換

ここではLLMを使わず、`intervention_plan` を実際に教師が言う全体向け発話と個別向け発話に変換します。

In [ ]:
import importlib
from pprint import pprint

import src.teacher.utterance_builder as utterance_builder

importlib.reload(utterance_builder)
RuleBasedTeacherUtteranceBuilder = utterance_builder.RuleBasedTeacherUtteranceBuilder

teacher_utterance_plan = RuleBasedTeacherUtteranceBuilder().build(intervention_plan)

print("全体向け発話:")
print(teacher_utterance_plan["whole_class_utterance"])
print("\n個別向け発話:")
for item in teacher_utterance_plan["individual_utterances"]:
    print(f"- {item['student_id']} ({item['support_type']}): {item['utterance']}")

pprint(teacher_utterance_plan)

## 11. 教師AI用コンテキストを作成

教師AI用コンテキストには、生徒状態、単元目標、直近の生徒発話、伝達AIの個別観察、クラス全体の要約、教師側の推定値、次に出せる問題が入ります。

In [ ]:
import importlib
from pprint import pprint

from src.state_manager import StateManager
import src.teacher.context_builder as teacher_context_builder

importlib.reload(teacher_context_builder)
TeacherContextBuilder = teacher_context_builder.TeacherContextBuilder

state_manager = StateManager()
student_state = state_manager.load_student(STUDENT_ID)

context_builder = TeacherContextBuilder()
teacher_context = context_builder.build_context(
    student_state=student_state,
    recent_student_utterance=student_utterance,
    communication_observation=observation,
    classroom_observation=classroom_observation,
    teacher_belief=teacher_belief,
)

pprint({
    "target_skill": teacher_context["target_skill"],
    "lesson_goal": teacher_context["lesson_goal"],
    "student_state_summary": teacher_context["student_state_summary"],
    "communication_ai_observation": teacher_context["communication_ai_observation"],
})

## 12. 授業方略を選択

現在のMVPでは、判断理由を確認しやすいようにルールベースで授業方略を選びます。将来的には同じコンテキストをLLM教師に渡せます。

In [ ]:
from pprint import pprint

from src.teacher import RuleBasedTeachingStrategySelector

selector = RuleBasedTeachingStrategySelector()
teacher_decision = selector.select_strategy(teacher_context)

pprint(teacher_decision)

## 13. 最新結果を保存

あとで比較できるように、教師AI用コンテキストと授業方略の判断結果をJSONで保存します。

In [ ]:
import json
from pathlib import Path

output_dir = Path("data/assessments")
output_dir.mkdir(parents=True, exist_ok=True)

context_path = output_dir / "teacher_context_latest.json"
decision_path = output_dir / "teaching_strategy_decision_latest.json"
events_path = output_dir / "observable_events_latest.json"
beliefs_path = output_dir / "teacher_beliefs_latest.json"
belief_table_path = output_dir / "teacher_belief_table_latest.csv"
belief_progress_path = output_dir / "teacher_belief_progress_validation.csv"
validation_events_path = output_dir / "observable_events_validation.json"
intervention_plan_path = output_dir / "intervention_plan_latest.json"
teacher_utterance_plan_path = output_dir / "teacher_utterance_plan_latest.json"

context_path.write_text(json.dumps(teacher_context, ensure_ascii=False, indent=2), encoding="utf-8")
decision_path.write_text(json.dumps(teacher_decision, ensure_ascii=False, indent=2), encoding="utf-8")
events_path.write_text(json.dumps(observable_events, ensure_ascii=False, indent=2), encoding="utf-8")
beliefs_path.write_text(json.dumps(teacher_beliefs, ensure_ascii=False, indent=2), encoding="utf-8")
belief_table.to_csv(belief_table_path, index=False)
belief_progress_table.to_csv(belief_progress_path, index=False)
validation_events_path.write_text(json.dumps(validation_events_all, ensure_ascii=False, indent=2), encoding="utf-8")
intervention_plan_path.write_text(json.dumps(intervention_plan, ensure_ascii=False, indent=2), encoding="utf-8")
teacher_utterance_plan_path.write_text(json.dumps(teacher_utterance_plan, ensure_ascii=False, indent=2), encoding="utf-8")

print("saved:", context_path)
print("saved:", decision_path)
print("saved:", events_path)
print("saved:", beliefs_path)
print("saved:", belief_table_path)
print("saved:", belief_progress_path)
print("saved:", validation_events_path)
print("saved:", intervention_plan_path)
print("saved:", teacher_utterance_plan_path)